<a href="https://colab.research.google.com/github/ehsankarami1358/LOKA_HYDRO/blob/main/HYDROPOWER_OPTIMIZATION_WORKFLOW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ============================================================
# HYDROPOWER OPTIMIZATION WORKFLOW
# Step 1: Turbine efficiency model from historian
# Step 2: Flow forecast model
# Step 3: Maximum possible production each day
# Step 4: Compare with actual production plan
# Step 5: Lost energy due to planning
# ============================================================

import re
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [9]:
# ============================================================
# 0) FILE PATHS
# ============================================================
HISTORIAN_FILE = "u2_MW_OP_FL_L_10_2_2025_17_3_2025_NEW_R3.csv"
PRODUCTION_FILE = "Production_program.xlsx"
WATER_FILE = "Water_change.xlsx"
NETHEAD_FILE = "Net_Head.xlsx"
HILLCHART_FILE = "LOKA_Digital_hillchart_extracted.xlsx"

In [10]:

# ============================================================
# 1) CONSTANTS
# ============================================================
RHO = 1000.0   # kg/m3
G = 9.81       # m/s2

# Assumption:
# 3 big units behave similarly to U2 model
NUM_BIG_UNITS = 3

In [11]:


# ============================================================
# 2) HELPER FUNCTIONS
# ============================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\xa0", " ").replace("\n", " ").strip()

def parse_number(x):
    if pd.isna(x):
        return np.nan

    s = clean_text(x)
    if s in ["", "-", "—", "nan", "None"]:
        return np.nan

    s = s.replace("m³/sn", "").replace("m3/sn", "").replace("m3/s", "")
    s = s.replace("m³/h", "").replace("m3/h", "")
    s = s.replace("MWh", "").replace("MW", "").replace("kWh", "")
    s = s.replace("%", "").replace("°C", "")
    s = s.replace(",", ".")
    s = re.sub(r"[^0-9\.\-]", "", s)

    if s in ["", ".", "-", "-.", ".-"]:
        return np.nan

    try:
        return float(s)
    except:
        return np.nan

In [12]:
# ============================================================
# 3) STEP 1 - PARSE DIGITAL HILL CHART
# ============================================================
def parse_loka_hillchart(excel_path):
    """
    Expected repeated block structure:
      NetHead_m=75
      ActivePower_MW ...
      Flow_m3s ...
      Efficency ...
      Oppening ...
    """
    raw = pd.read_excel(excel_path, header=0)
    raw = raw.reset_index(drop=True)

    hill_rows = []
    i = 0

    while i < len(raw):
        cell = clean_text(raw.iloc[i, 0])

        m = re.match(r"NetHead_m\s*=\s*([0-9.]+)", cell)
        if m:
            net_head = float(m.group(1))

            if i + 3 < len(raw):
                power_row = raw.iloc[i + 1, :]
                flow_row = raw.iloc[i + 2, :]
                eta_row = raw.iloc[i + 3, :]
                opening_row = raw.iloc[i + 4, :] if i + 4 < len(raw) else None

                for col in raw.columns[1:]:
                    power = pd.to_numeric(power_row[col], errors="coerce")
                    flow = pd.to_numeric(flow_row[col], errors="coerce")
                    eta = pd.to_numeric(eta_row[col], errors="coerce")
                    opening = pd.to_numeric(opening_row[col], errors="coerce") if opening_row is not None else np.nan

                    if pd.notna(power) and pd.notna(flow) and pd.notna(eta):
                        hill_rows.append({
                            "net_head_m": net_head,
                            "power_mw": float(power),
                            "flow_m3s": float(flow),
                            "eta": float(eta),
                            "opening_pct": float(opening) if pd.notna(opening) else np.nan
                        })
            i += 5
        else:
            i += 1

    hill_df = pd.DataFrame(hill_rows).sort_values(["net_head_m", "power_mw"]).reset_index(drop=True)
    return hill_df


def interp_by_head_and_power(hill_df, head, power, value_col):
    heads = np.sort(hill_df["net_head_m"].unique())

    if head <= heads.min():
        h1 = h2 = heads.min()
    elif head >= heads.max():
        h1 = h2 = heads.max()
    else:
        h1 = heads[heads <= head].max()
        h2 = heads[heads >= head].min()

    df1 = hill_df[hill_df["net_head_m"] == h1].sort_values("power_mw")
    v1 = np.interp(power, df1["power_mw"], df1[value_col])

    if h1 == h2:
        return v1

    df2 = hill_df[hill_df["net_head_m"] == h2].sort_values("power_mw")
    v2 = np.interp(power, df2["power_mw"], df2[value_col])

    return np.interp(head, [h1, h2], [v1, v2])


def flow_from_power_head(hill_df, power_mw, net_head_m):
    return interp_by_head_and_power(hill_df, net_head_m, power_mw, "flow_m3s")


def eta_from_power_head(hill_df, power_mw, net_head_m):
    return interp_by_head_and_power(hill_df, net_head_m, power_mw, "eta")


def opening_from_power_head(hill_df, power_mw, net_head_m):
    if "opening_pct" not in hill_df.columns or hill_df["opening_pct"].isna().all():
        return np.nan
    return interp_by_head_and_power(hill_df, net_head_m, power_mw, "opening_pct")


def power_from_flow_head(hill_df, flow_m3s, net_head_m):
    candidate_powers = np.linspace(hill_df["power_mw"].min(), hill_df["power_mw"].max(), 500)
    candidate_flows = np.array([flow_from_power_head(hill_df, p, net_head_m) for p in candidate_powers])

    sort_idx = np.argsort(candidate_flows)
    candidate_flows = candidate_flows[sort_idx]
    candidate_powers = candidate_powers[sort_idx]

    return np.interp(flow_m3s, candidate_flows, candidate_powers)

In [13]:

    print("STEP 1 - Load hill chart...")
    hill_df = parse_loka_hillchart(HILLCHART_FILE)
    print("Hill chart rows:", len(hill_df))
    print("Head slices:", sorted(hill_df["net_head_m"].unique()))



STEP 1 - Load hill chart...
Hill chart rows: 55
Head slices: [np.float64(77.5), np.float64(80.0), np.float64(82.5), np.float64(85.0), np.float64(87.5)]


In [14]:

# ============================================================
# 4) STEP 1 - LOAD HISTORIAN AND BUILD TURBINE MODEL
# ============================================================
def load_historian(csv_path):
    hist = pd.read_csv(csv_path)

    rename_map = {
        "Timestamp": "timestamp",
        "ACTIVE_POWER(MW)": "power_mw",
        "OPPENING(%)": "opening_pct",
        "FLOW(m3/s)": "flow_m3s",
        "HEAD_L(m)": "head_l_m",
        "TAIL_L(m)": "tail_l_m",
        "SPEED(RPM)": "speed_rpm"
    }

    hist = hist.rename(columns=rename_map)
    hist["timestamp"] = pd.to_datetime(hist["timestamp"], errors="coerce")
    hist = hist.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

    hist["net_head_m"] = hist["head_l_m"] - hist["tail_l_m"]

    hist["is_running"] = (
        (hist["power_mw"] > 1) &
        (hist["flow_m3s"] > 10) &
        (hist["net_head_m"] > 1)
    )

    if "speed_rpm" in hist.columns:
        hist["is_running"] = hist["is_running"] & (hist["speed_rpm"] > 100)

    run = hist[hist["is_running"]].copy()
    run["power_w"] = run["power_mw"] * 1e6
    run["eta_actual"] = run["power_w"] / (RHO * G * run["flow_m3s"] * run["net_head_m"])
    run = run[(run["eta_actual"] > 0.40) & (run["eta_actual"] < 1.05)].copy()

    run["date"] = run["timestamp"].dt.floor("D")
    return hist, run


def build_turbine_efficiency_model(run_df):
    """
    Actual turbine model from historian:
      power_mw = f(flow_m3s, net_head_m, opening_pct)
    """
    features = ["flow_m3s", "net_head_m", "opening_pct"]
    data = run_df.dropna(subset=features + ["power_mw"]).copy()

    X = data[features]
    y = data["power_mw"]

    split_idx = max(int(len(data) * 0.8), 1)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    metrics = {
        "rows_used": len(data),
        "mae_mw": mean_absolute_error(y_test, pred) if len(y_test) else np.nan,
        "r2": r2_score(y_test, pred) if len(y_test) else np.nan,
        "eta_mean": data["eta_actual"].mean(),
        "eta_min": data["eta_actual"].min(),
        "eta_max": data["eta_actual"].max(),
    }

    return model, metrics, data


In [15]:
print("\nSTEP 1 - Load historian...")
hist_all, hist_run = load_historian(HISTORIAN_FILE)
print("Historian running rows:", len(hist_run))

print("\nSTEP 1 - Build turbine efficiency model...")
turbine_model, turbine_metrics, turbine_train_df = build_turbine_efficiency_model(hist_run)
print("Turbine model metrics:", turbine_metrics)

# Compare historian eta with hill-chart eta at nearest operating point
hist_run["eta_hill"] = hist_run.apply(
    lambda r: eta_from_power_head(hill_df, r["power_mw"], r["net_head_m"]), axis=1
)
hist_run["eta_gap_actual_minus_hill"] = hist_run["eta_actual"] - hist_run["eta_hill"]


STEP 1 - Load historian...
Historian running rows: 2020

STEP 1 - Build turbine efficiency model...
Turbine model metrics: {'rows_used': 2020, 'mae_mw': 4.578227517952143, 'r2': 0.6352070290449512, 'eta_mean': np.float64(0.9043506793453326), 'eta_min': 0.5491274412284046, 'eta_max': 0.9354337308101012}


In [16]:
# ============================================================
# 5) STEP 2 - LOAD WATER FILE
# ============================================================
def load_water_file(excel_path):
    water = pd.read_excel(excel_path)
    water.columns = [clean_text(c) for c in water.columns]

    rename_map = {
        "TARİH": "date",
        "GELEN SUYUN DEBİSİ (m3/sn)": "inflow_m3s",
        "GELEN SUYUN DEBİSİ (m³/sn)": "inflow_m3s",
        "GELEN SU (m3)": "inflow_volume_m3",
        "AKTİF GÖL HACMİ (m3)": "active_storage_m3",
        "AKTİF GÖL HACMİ  (m3)": "active_storage_m3",
        "GÖLDEKİ HACİM FARKI (m3)": "storage_change_file_m3",
        "DSİ BIRAKILAN SU (m3/h)": "dsi_release_m3h",
        "ÜRETİLEBİLECEK ENERJİ (kWh)": "potential_energy_kwh",
        "ÜRETİLEBİLECEK ENERJİ ( kWh)": "potential_energy_kwh",
        "ENERJİYE HARCANAN SU (m3)": "water_used_for_generation_m3",
        "SUYUN ENERJİYE ORANI (m3/kWh)": "water_energy_ratio_m3_per_kwh"
    }

    water = water.rename(columns=rename_map)

    if "date" in water.columns:
        water["date"] = pd.to_datetime(water["date"], errors="coerce")
        water = water.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    else:
        raise ValueError("Date column not found in water file.")

    # parse numeric columns
    for col in water.columns:
        if col != "date":
            water[col] = water[col].apply(parse_number)

    if "active_storage_m3" in water.columns:
        water["storage_change_calc_m3"] = water["active_storage_m3"].diff()
    else:
        water["storage_change_calc_m3"] = np.nan

    water["storage_change_m3"] = water.get("storage_change_file_m3", np.nan)
    water["storage_change_m3"] = water["storage_change_m3"].fillna(water["storage_change_calc_m3"])

    if "dsi_release_m3h" in water.columns:
        water["dsi_release_m3s"] = water["dsi_release_m3h"] / 3600.0
    else:
        water["dsi_release_m3s"] = 0.0

    water["effective_inflow_m3s"] = water["inflow_m3s"] - water["dsi_release_m3s"]
    water["effective_inflow_m3s"] = water["effective_inflow_m3s"].clip(lower=0)

    return water

In [17]:

    # ------------------------
    print("\nSTEP 2 - Load water file...")
    water_df = load_water_file(WATER_FILE)
    print("Water rows:", len(water_df))
    water_df


STEP 2 - Load water file...
Water rows: 31


,date,24:00 GÖL SEVİYESİ,inflow_m3s,inflow_volume_m3,water_energy_ratio_m3_per_kwh,water_used_for_generation_m3,DOLU SAVAKTAN BIRAKILAN SU (m3),DOLU SAVAKTAN BIRAKILAN SU İLE ÜRETİLEBİLECEK ENERJİ ( kWh ),storage_change_file_m3,AÇIKLAMA,...,potential_energy_kwh,ÖLÜ HACİM (m3),GÖL HACMİ (m3),Y HES GELEN SU (m3/sn),dsi_release_m3h,….. GELEN SU (m3/sn),storage_change_calc_m3,storage_change_m3,dsi_release_m3s,effective_inflow_m3s
0,2026-02-01,1094.6294,428.289037,3.700417e+07,4.824793,2.317254e+07,0.0,0.0,1.372924e+07,NaN,...,2.336945e+07,3.144816e+08,4.272344e+08,433.130,0.0,NaN,NaN,1.372924e+07,0.0,428.289037
1,2026-02-02,1094.9216,188.255051,1.626524e+07,4.821241,1.234097e+07,0.0,0.0,3.821863e+06,NaN,...,2.494279e+07,3.144816e+08,4.347368e+08,143.560,0.0,NaN,7.502465e+06,3.821863e+06,0.0,188.255051
2,2026-02-03,1095.2736,203.635388,1.759410e+07,4.750280,1.255035e+07,0.0,0.0,4.941343e+06,NaN,...,2.612397e+07,3.144816e+08,4.385778e+08,143.000,0.0,NaN,3.840972e+06,4.941343e+06,0.0,203.635388
3,2026-02-04,1096.5500,405.106540,3.500121e+07,4.677331,1.656814e+07,0.0,0.0,1.833066e+07,NaN,...,2.910362e+07,3.144816e+08,4.506089e+08,371.330,0.0,NaN,1.203109e+07,1.833066e+07,0.0,405.106540
4,2026-02-05,1096.6719,320.094734,2.765619e+07,4.680301,2.581849e+07,0.0,0.0,1.735290e+06,NaN,...,3.126063e+07,3.144816e+08,4.607908e+08,300.440,0.0,NaN,1.018191e+07,1.735290e+06,0.0,320.094734
5,2026-02-06,1096.2346,280.590581,2.424303e+07,4.676072,3.036480e+07,0.0,0.0,-6.224175e+06,NaN,...,3.164243e+07,3.144816e+08,4.624439e+08,267.360,0.0,NaN,1.653094e+06,-6.224175e+06,0.0,280.590581
6,2026-02-07,1096.3435,444.118828,3.837187e+07,4.683257,3.671084e+07,0.0,0.0,1.558632e+06,NaN,...,3.024859e+07,3.144816e+08,4.561435e+08,434.500,0.0,NaN,-6.300364e+06,1.558632e+06,0.0,444.118828
7,2026-02-08,1096.1721,428.004048,3.697955e+07,4.680000,3.933031e+07,0.0,0.0,-2.453163e+06,NaN,...,3.049259e+07,3.144816e+08,4.571869e+08,411.950,0.0,NaN,1.043382e+06,-2.453163e+06,0.0,428.004048
8,2026-02-09,1094.6448,220.468889,1.904851e+07,4.750004,4.036768e+07,0.0,0.0,-2.142156e+07,NaN,...,2.862069e+07,3.144816e+08,4.504300e+08,196.140,0.0,NaN,-6.756933e+06,-2.142156e+07,0.0,220.468889
9,2026-02-10,1093.0839,254.035181,2.194864e+07,4.900000,4.226221e+07,0.0,0.0,-2.041597e+07,NaN,...,2.320402e+07,3.144816e+08,4.281813e+08,196.400,0.0,NaN,-2.224866e+07,-2.041597e+07,0.0,254.035181


In [18]:
# ============================================================
# 6) STEP 2 - LOAD NET HEAD FILE
# ============================================================
def load_net_head_file(excel_path):
    head = pd.read_excel(excel_path)
    head.columns = [clean_text(c) for c in head.columns]

    rename_map = {
        "Timestamp": "timestamp",
        "Net_Head": "net_head_m",
        "Net Head": "net_head_m",
        "NET_HEAD": "net_head_m"
    }
    head = head.rename(columns=rename_map)

    head["timestamp"] = pd.to_datetime(head["timestamp"], errors="coerce")
    head["net_head_m"] = pd.to_numeric(head["net_head_m"], errors="coerce")
    head = head.dropna(subset=["timestamp", "net_head_m"]).sort_values("timestamp").reset_index(drop=True)

    head["date"] = head["timestamp"].dt.floor("D")

    daily_head = head.groupby("date", as_index=False).agg(
        net_head_mean_m=("net_head_m", "mean"),
        net_head_min_m=("net_head_m", "min"),
        net_head_max_m=("net_head_m", "max"),
        net_head_std_m=("net_head_m", "std")
    )
    daily_head["head_change_m"] = daily_head["net_head_mean_m"].diff()

    return head, daily_head

In [19]:
    print("\nSTEP 2 - Load net head file...")
    _, daily_head_df = load_net_head_file(NETHEAD_FILE)
    print("Daily net head rows:", len(daily_head_df))


STEP 2 - Load net head file...
Daily net head rows: 26


In [20]:
# ============================================================
# 7) STEP 2 - FLOW FORECAST MODEL
# ============================================================
def build_flow_forecast_dataset(water_df):
    df = water_df.copy().sort_values("date").reset_index(drop=True)

    df["inflow_lag1"] = df["effective_inflow_m3s"].shift(1)
    df["inflow_lag2"] = df["effective_inflow_m3s"].shift(2)
    df["inflow_roll3"] = df["effective_inflow_m3s"].rolling(3).mean()
    df["storage_change_lag1"] = df["storage_change_m3"].shift(1)
    df["month"] = df["date"].dt.month
    df["dayofyear"] = df["date"].dt.dayofyear

    df["target_next_day_inflow"] = df["effective_inflow_m3s"].shift(-1)

    features = [
        "effective_inflow_m3s",
        "inflow_lag1",
        "inflow_lag2",
        "inflow_roll3",
        "active_storage_m3",
        "storage_change_m3",
        "storage_change_lag1",
        "month",
        "dayofyear"
    ]

    model_df = df.dropna(subset=features + ["target_next_day_inflow"]).copy()
    return model_df, features


def train_flow_forecast_model(flow_df, features):
    X = flow_df[features]
    y = flow_df["target_next_day_inflow"]

    split_idx = max(int(len(flow_df) * 0.8), 1)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = RandomForestRegressor(
        n_estimators=250,
        max_depth=10,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    metrics = {
        "rows_used": len(flow_df),
        "mae_m3s": mean_absolute_error(y_test, pred) if len(y_test) else np.nan,
        "r2": r2_score(y_test, pred) if len(y_test) else np.nan
    }

    flow_df = flow_df.copy()
    flow_df["pred_next_day_inflow_m3s"] = model.predict(flow_df[features])

    return model, metrics, flow_df


In [21]:
    print("\nSTEP 2 - Build flow forecast dataset...")
    flow_ds, flow_features = build_flow_forecast_dataset(water_df)
    print("Flow dataset rows:", len(flow_ds))

    print("\nSTEP 2 - Train flow forecast model...")
    flow_model, flow_metrics, flow_scored = train_flow_forecast_model(flow_ds, flow_features)
    print("Flow model metrics:", flow_metrics)


STEP 2 - Build flow forecast dataset...
Flow dataset rows: 25

STEP 2 - Train flow forecast model...
Flow model metrics: {'rows_used': 25, 'mae_m3s': 151.75434757405924, 'r2': -2.1364265512850364}


In [22]:

# ============================================================
# 8) STEP 3 - MAXIMUM POSSIBLE PRODUCTION EACH DAY
# ============================================================
def estimate_daily_max_possible_production(water_df, daily_head_df, hill_df):
    """
    Estimate maximum feasible production each day for 3 big units.
    Uses:
    - effective inflow
    - real daily net head
    - hill chart
    Assumes sustainable water-neutral operation for that day.
    """
    df = water_df.merge(daily_head_df, on="date", how="left").sort_values("date").reset_index(drop=True)
    df["net_head_mean_m"] = df["net_head_mean_m"].interpolate().bfill().ffill()
    df["head_change_m"] = df["head_change_m"].fillna(0.0)

    rows = []

    for _, row in df.iterrows():
        head = row["net_head_mean_m"]
        q_available = row["effective_inflow_m3s"]

        q_per_unit = q_available / NUM_BIG_UNITS
        unit_power_max = power_from_flow_head(hill_df, q_per_unit, head)

        unit_power_max = float(np.clip(
            unit_power_max,
            hill_df["power_mw"].min(),
            hill_df["power_mw"].max()
        ))

        unit_flow = flow_from_power_head(hill_df, unit_power_max, head)
        unit_eta = eta_from_power_head(hill_df, unit_power_max, head)
        unit_opening = opening_from_power_head(hill_df, unit_power_max, head)

        total_big_power_mw = NUM_BIG_UNITS * unit_power_max
        total_big_flow_m3s = NUM_BIG_UNITS * unit_flow
        max_big_daily_mwh = total_big_power_mw * 24

        rows.append({
            "date": row["date"],
            "effective_inflow_m3s": q_available,
            "net_head_mean_m": head,
            "unit_power_max_mw": unit_power_max,
            "unit_flow_at_max_m3s": unit_flow,
            "unit_eta_at_max": unit_eta,
            "unit_opening_at_max_pct": unit_opening,
            "max_total_big_power_mw": total_big_power_mw,
            "max_total_big_flow_m3s": total_big_flow_m3s,
            "max_big_daily_mwh": max_big_daily_mwh,
            "water_balance_gap_m3s": q_available - total_big_flow_m3s
        })

    return pd.DataFrame(rows)

In [23]:
    # Step 3
    # ------------------------
    print("\nSTEP 3 - Estimate maximum possible daily production...")
    max_possible_df = estimate_daily_max_possible_production(water_df, daily_head_df, hill_df)
    print("Max possible rows:", len(max_possible_df))
    max_possible_df


STEP 3 - Estimate maximum possible daily production...
Max possible rows: 31


,date,effective_inflow_m3s,net_head_mean_m,unit_power_max_mw,unit_flow_at_max_m3s,unit_eta_at_max,unit_opening_at_max_pct,max_total_big_power_mw,max_total_big_flow_m3s,max_big_daily_mwh,water_balance_gap_m3s
0,2026-02-01,428.289037,84.101602,111.207772,142.763012,0.949416,NaN,333.623315,428.289037,8006.959568,0.000000e+00
1,2026-02-02,188.255051,85.114282,80.000000,107.954287,0.904771,NaN,240.000000,323.862861,5760.000000,-1.356078e+02
2,2026-02-03,203.635388,85.387602,80.000000,107.844959,0.904225,NaN,240.000000,323.534877,5760.000000,-1.198995e+02
3,2026-02-04,405.106540,86.050415,107.136343,135.035513,0.941788,NaN,321.409030,405.106540,7713.816713,0.000000e+00
4,2026-02-05,320.094734,86.118009,80.000000,107.552796,0.902764,NaN,240.000000,322.658389,5760.000000,-2.563656e+00
5,2026-02-06,280.590581,85.864404,80.000000,107.654239,0.903271,NaN,240.000000,322.962716,5760.000000,-4.237213e+01
6,2026-02-07,444.118828,85.416415,119.033536,148.039609,0.957806,NaN,357.100608,444.118828,8570.414601,-5.684342e-14
7,2026-02-08,428.004048,85.330821,114.028792,142.668016,0.955058,NaN,342.086376,428.004048,8210.073034,-1.136868e-13
8,2026-02-09,220.468889,84.600853,80.000000,108.319318,0.904202,NaN,240.000000,324.957954,5760.000000,-1.044891e+02
9,2026-02-10,254.035181,83.125888,80.000000,109.499289,0.901252,NaN,240.000000,328.497868,5760.000000,-7.446269e+01


In [24]:
# ============================================================
# 9) STEP 4 - PARSE PRODUCTION PROGRAM
# ============================================================
def find_program_anchor_column(df):
    for col in df.columns:
        if df[col].astype(str).str.contains("GÖP ÜRETİM PROGRAMI", case=False, na=False).any():
            return col
    return None

def extract_first_date_from_block(block_df):
    date_pattern = re.compile(r"(\d{2}\.\d{2}\.\d{4})")

    for _, row in block_df.iterrows():
        for val in row.values:
            s = clean_text(val)
            m = date_pattern.search(s)
            if m:
                dt = pd.to_datetime(m.group(1), format="%d.%m.%Y", errors="coerce")
                if pd.notna(dt):
                    # Usually tomorrow date is written; subtract 1 day to get report date
                    return dt - pd.Timedelta(days=1)
    return pd.NaT

def parse_production_program(excel_path):
    """
    Parses daily unit plan from Production_program.xlsx
    """
    xls = pd.ExcelFile(excel_path)
    all_rows = []

    for sheet in xls.sheet_names:
        df = pd.read_excel(excel_path, sheet_name=sheet, header=None)
        df = df.applymap(clean_text)

        anchor_col = find_program_anchor_column(df)
        if anchor_col is None:
            continue

        start_rows = df.index[
            df[anchor_col].astype(str).str.contains("GÖP ÜRETİM PROGRAMI", case=False, na=False)
        ].tolist()

        for start in start_rows:
            block = df.iloc[start:start+25].copy()
            block_date = extract_first_date_from_block(block)

            c_name = anchor_col
            c_today_hours = anchor_col + 1
            c_today_power = anchor_col + 2
            c_tom_hours = anchor_col + 3
            c_tom_power = anchor_col + 4

            required_cols = [c_name, c_today_hours, c_today_power, c_tom_hours, c_tom_power]
            if not all(c in df.columns for c in required_cols):
                continue

            idx_u1 = idx_u2 = idx_u3 = idx_u4 = idx_total = None

            for idx in block.index:
                label = clean_text(df.loc[idx, c_name])

                if "Büyük Ünite (U1)" in label:
                    idx_u1 = idx
                elif "Büyük Ünite (U2)" in label:
                    idx_u2 = idx
                elif "Büyük Ünite (U3)" in label:
                    idx_u3 = idx
                elif "Küçük Ünite (U4)" in label or "Kucuk Unite (U4)" in label:
                    idx_u4 = idx
                elif "TOPLAM" in label.upper():
                    idx_total = idx

            def read_unit(idx):
                if idx is None:
                    return np.nan, np.nan, np.nan
                hrs = parse_number(df.loc[idx, c_today_hours])
                pwr = parse_number(df.loc[idx, c_today_power])
                mwh = hrs * pwr if pd.notna(hrs) and pd.notna(pwr) else np.nan
                return hrs, pwr, mwh

            u1_h, u1_p, u1_mwh = read_unit(idx_u1)
            u2_h, u2_p, u2_mwh = read_unit(idx_u2)
            u3_h, u3_p, u3_mwh = read_unit(idx_u3)
            u4_h, u4_p, u4_mwh = read_unit(idx_u4)

            plant_total_today_mwh = np.nan
            if idx_total is not None:
                plant_total_today_mwh = parse_number(df.loc[idx_total, c_today_hours])

            all_rows.append({
                "date": block_date,
                "sheet": sheet,
                "u1_plan_mw": u1_p,
                "u2_plan_mw": u2_p,
                "u3_plan_mw": u3_p,
                "u4_plan_mw": u4_p,
                "u1_plan_mwh": u1_mwh,
                "u2_plan_mwh": u2_mwh,
                "u3_plan_mwh": u3_mwh,
                "u4_plan_mwh": u4_mwh,
                "big_units_plan_total_mwh": np.nansum([u1_mwh, u2_mwh, u3_mwh]),
                "plant_plan_total_mwh_from_units": np.nansum([u1_mwh, u2_mwh, u3_mwh, u4_mwh]),
                "plant_total_today_mwh_reported": plant_total_today_mwh
            })

    out = pd.DataFrame(all_rows)
    out = out.dropna(subset=["date"]).sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)
    return out


In [26]:

# ============================================================
# 10) STEP 4 - ACTUAL ENERGY FROM HISTORIAN
# ============================================================
def build_daily_actual_from_historian(run_df):
    df = run_df.copy().sort_values("timestamp").reset_index(drop=True)
    dt_seconds = df["timestamp"].diff().dt.total_seconds().dropna()

    if len(dt_seconds) == 0:
        sample_hours = 1 / 60
    else:
        sample_hours = dt_seconds.median() / 3600.0

    daily = df.groupby("date", as_index=False).agg(
        u2_actual_power_mean_mw=("power_mw", "mean"),
        u2_actual_flow_mean_m3s=("flow_m3s", "mean"),
        u2_actual_eta_mean=("eta_actual", "mean"),
        samples=("power_mw", "count")
    )
    daily["u2_running_hours"] = daily["samples"] * sample_hours
    daily["u2_actual_mwh"] = daily["u2_actual_power_mean_mw"] * daily["u2_running_hours"]

    # If 3 big units are assumed similar and all running similarly:
    daily["three_big_units_actual_estimated_mwh"] = daily["u2_actual_mwh"] * 3

    return daily, sample_hours

In [27]:

# ============================================================
# 11) STEP 4 + 5 - COMPARE PLAN VS MAX POSSIBLE VS ACTUAL
# ============================================================
def compare_plan_vs_possible_vs_actual(plan_df, max_df, actual_df):
    df = plan_df.merge(max_df, on="date", how="inner")
    df = df.merge(actual_df[["date", "u2_actual_mwh", "three_big_units_actual_estimated_mwh"]], on="date", how="left")

    # Big unit comparison only
    df["planning_gap_mwh"] = df["max_big_daily_mwh"] - df["big_units_plan_total_mwh"]
    df["execution_gap_mwh"] = df["big_units_plan_total_mwh"] - df["three_big_units_actual_estimated_mwh"]
    df["total_lost_opportunity_mwh"] = df["max_big_daily_mwh"] - df["three_big_units_actual_estimated_mwh"]

    # Lost energy due to planning
    df["lost_energy_due_to_planning_mwh"] = df["planning_gap_mwh"]
    df["lost_energy_due_to_planning_pct"] = (
        df["lost_energy_due_to_planning_mwh"] / df["max_big_daily_mwh"] * 100.0
    )

    return df

In [29]:
# ============================================================
# 12) MAIN RUNNER
# ============================================================
def main():
    out_dir = Path("hydro_optimization_outputs")
    out_dir.mkdir(exist_ok=True)

    # ------------------------
    # Step 1
    # ------------------------
    print("STEP 1 - Load hill chart...")
    hill_df = parse_loka_hillchart(HILLCHART_FILE)
    print("Hill chart rows:", len(hill_df))
    print("Head slices:", sorted(hill_df["net_head_m"].unique()))

    print("\nSTEP 1 - Load historian...")
    hist_all, hist_run = load_historian(HISTORIAN_FILE)
    print("Historian running rows:", len(hist_run))

    print("\nSTEP 1 - Build turbine efficiency model...")
    turbine_model, turbine_metrics, turbine_train_df = build_turbine_efficiency_model(hist_run)
    print("Turbine model metrics:", turbine_metrics)

    # Compare historian eta with hill-chart eta at nearest operating point
    hist_run["eta_hill"] = hist_run.apply(
        lambda r: eta_from_power_head(hill_df, r["power_mw"], r["net_head_m"]), axis=1
    )
    hist_run["eta_gap_actual_minus_hill"] = hist_run["eta_actual"] - hist_run["eta_hill"]

    # ------------------------
    # Step 2
    # ------------------------
    print("\nSTEP 2 - Load water file...")
    water_df = load_water_file(WATER_FILE)
    print("Water rows:", len(water_df))

    print("\nSTEP 2 - Load net head file...")
    _, daily_head_df = load_net_head_file(NETHEAD_FILE)
    print("Daily net head rows:", len(daily_head_df))

    print("\nSTEP 2 - Build flow forecast dataset...")
    flow_ds, flow_features = build_flow_forecast_dataset(water_df)
    print("Flow dataset rows:", len(flow_ds))

    print("\nSTEP 2 - Train flow forecast model...")
    flow_model, flow_metrics, flow_scored = train_flow_forecast_model(flow_ds, flow_features)
    print("Flow model metrics:", flow_metrics)

    # ------------------------
    # Step 3
    # ------------------------
    print("\nSTEP 3 - Estimate maximum possible daily production...")
    max_possible_df = estimate_daily_max_possible_production(water_df, daily_head_df, hill_df)
    print("Max possible rows:", len(max_possible_df))

    # ------------------------
    # Step 4
    # ------------------------
    print("\nSTEP 4 - Parse production plan...")
    plan_df = parse_production_program(PRODUCTION_FILE)
    print("Plan rows:", len(plan_df))

    print("\nSTEP 4 - Build actual daily energy from historian...")
    actual_df, sample_hours = build_daily_actual_from_historian(hist_run)
    print("Actual rows:", len(actual_df))
    print("Sample interval hours:", sample_hours)

    print("\nSTEP 4 - Compare plan vs possible vs actual...")
    comparison_df = compare_plan_vs_possible_vs_actual(plan_df, max_possible_df, actual_df)
    print("Comparison rows:", len(comparison_df))

    # ------------------------
    # Step 5
    # ------------------------
    print("\nSTEP 5 - Calculate lost energy due to planning...")
    if len(comparison_df) > 0:
        total_lost_planning = comparison_df["lost_energy_due_to_planning_mwh"].sum()
        avg_lost_planning = comparison_df["lost_energy_due_to_planning_mwh"].mean()

        print(f"Total lost energy due to planning (MWh): {total_lost_planning:.2f}")
        print(f"Average lost energy due to planning (MWh/day): {avg_lost_planning:.2f}")

        print("\nTop 10 days with highest lost energy due to planning:")
        cols = [
            "date",
            "big_units_plan_total_mwh",
            "max_big_daily_mwh",
            "three_big_units_actual_estimated_mwh",
            "lost_energy_due_to_planning_mwh",
            "lost_energy_due_to_planning_pct"
        ]
        print(
            comparison_df.sort_values("lost_energy_due_to_planning_mwh", ascending=False)
            .head(10)[cols].to_string(index=False)
        )
    # ------------------------
    # Save outputs
    # ------------------------
    hill_df.to_csv(out_dir / "01_hillchart_parsed.csv", index=False, encoding="utf-8-sig")
    hist_run.to_csv(out_dir / "02_historian_running_with_efficiency.csv", index=False, encoding="utf-8-sig")
    turbine_train_df.to_csv(out_dir / "03_turbine_training_data.csv", index=False, encoding="utf-8-sig")
    water_df.to_csv(out_dir / "04_water_daily_clean.csv", index=False, encoding="utf-8-sig")
    daily_head_df.to_csv(out_dir / "05_daily_net_head.csv", index=False, encoding="utf-8-sig")
    flow_scored.to_csv(out_dir / "06_flow_forecast_scored.csv", index=False, encoding="utf-8-sig")
    max_possible_df.to_csv(out_dir / "07_max_possible_daily_production.csv", index=False, encoding="utf-8-sig")
    plan_df.to_csv(out_dir / "08_production_plan_parsed.csv", index=False, encoding="utf-8-sig")
    actual_df.to_csv(out_dir / "09_actual_daily_energy_from_historian.csv", index=False, encoding="utf-8-sig")
    comparison_df.to_csv(out_dir / "10_plan_vs_possible_vs_actual.csv", index=False, encoding="utf-8-sig")

    print(f"\nAll files saved in: {out_dir.resolve()}")

if __name__ == "__main__":
    main()

STEP 1 - Load hill chart...
Hill chart rows: 55
Head slices: [np.float64(77.5), np.float64(80.0), np.float64(82.5), np.float64(85.0), np.float64(87.5)]

STEP 1 - Load historian...
Historian running rows: 2020

STEP 1 - Build turbine efficiency model...
Turbine model metrics: {'rows_used': 2020, 'mae_mw': 4.5782275179521426, 'r2': 0.6352070290449512, 'eta_mean': np.float64(0.9043506793453326), 'eta_min': 0.5491274412284046, 'eta_max': 0.9354337308101012}

STEP 2 - Load water file...
Water rows: 31

STEP 2 - Load net head file...
Daily net head rows: 26

STEP 2 - Build flow forecast dataset...
Flow dataset rows: 25

STEP 2 - Train flow forecast model...
Flow model metrics: {'rows_used': 25, 'mae_m3s': 151.75434757405924, 'r2': -2.1364265512850356}

STEP 3 - Estimate maximum possible daily production...
Max possible rows: 31

STEP 4 - Parse production plan...


/tmp/ipykernel_31728/3075640460.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)
/tmp/ipykernel_31728/3075640460.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)
/tmp/ipykernel_31728/3075640460.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)
/tmp/ipykernel_31728/3075640460.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)
/tmp/ipykernel_31728/3075640460.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)
/tmp/ipykernel_31728/3075640460.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)
/tmp/ipykernel_31728/3075640460.py:33: FutureWarning: DataFrame.applymap has been deprec

Plan rows: 354

STEP 4 - Build actual daily energy from historian...
Actual rows: 25
Sample interval hours: 0.25

STEP 4 - Compare plan vs possible vs actual...
Comparison rows: 0

STEP 5 - Calculate lost energy due to planning...

All files saved in: /content/hydro_optimization_outputs


/tmp/ipykernel_31728/3075640460.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)


In [30]:
plan_df

,date,sheet,u1_plan_mw,u2_plan_mw,u3_plan_mw,u4_plan_mw,u1_plan_mwh,u2_plan_mwh,u3_plan_mwh,u4_plan_mwh,big_units_plan_total_mwh,plant_plan_total_mwh_from_units,plant_total_today_mwh_reported
0,2024-12-24,12_2024,NaN,NaN,113.0,NaN,NaN,NaN,1808.0,NaN,1808.0,1808.0,4604.0
1,2024-12-25,12_2024,114.0,113.0,NaN,NaN,912.0,1808.0,NaN,NaN,2720.0,2720.0,2856.3
2,2024-12-26,12_2024,NaN,113.0,NaN,NaN,NaN,2712.0,NaN,NaN,2712.0,2712.0,2856.3
3,2024-12-27,12_2024,NaN,113.0,NaN,25.0,NaN,2712.0,NaN,600.0,2712.0,3312.0,712.3
4,2024-12-28,12_2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2780.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
349,2025-12-22,12_2025,107.0,107.0,107.0,NaN,2568.0,2568.0,642.0,NaN,5778.0,5778.0,3368.2
350,2025-12-23,12_2025,107.0,NaN,108.0,22.0,2568.0,NaN,648.0,396.0,3216.0,3612.0,1326.0
351,2025-12-27,12_2025,105.0,105.0,106.0,24.0,2520.0,1470.0,636.0,408.0,4626.0,5034.0,1726.0
352,2025-12-28,12_2025,105.0,105.0,106.0,NaN,2520.0,2520.0,636.0,NaN,5676.0,5676.0,4714.2
